# GitHub MCP Server

In [20]:
import os
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AzureOpenAI
from dotenv import load_dotenv
import json

In [10]:
load_dotenv()

True

In [16]:
async with AsyncExitStack() as stack:
    server_params = StdioServerParameters(
        command='docker',
        args=[
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        env={
            "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_TOKEN", ""),
        }
    )

    stdio_transport = await stack.enter_async_context(
        stdio_client(server_params)
    )

    stdio, write = stdio_transport
    session = await stack.enter_async_context(
        ClientSession(stdio, write)
    )

    await session.initialize()

    response = await session.list_tools()


In [21]:
with open("results/tool_list.json", "w", encoding="utf-8") as f:
    json.dump(response.model_dump(), f, ensure_ascii=False, indent=2)

In [32]:
def make_tool(tool):
    if tool["inputSchema"].get("required") == None:
        print(tool["name"])
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool["inputSchema"],
            # "parameters": {
            #     "type": "object",
            #     "properties": tool["inputSchema"]["properties"],
            #     "required": tool["inputSchema"].get("required")
            # }
        }
    }

In [33]:
tool_list = [make_tool(tool.model_dump()) for tool in response.tools]

get_me
list_notifications
mark_all_notifications_read


In [30]:
llm = AzureOpenAI(
    azure_endpoint=os.getenv("ENDPOINT_URL"),
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("API_VERSION"),
)

In [ ]:
response = llm.chat.completions.create(
    model=os.getenv("DEPLOYMENT_NAME"),
    messages=[
        {
            "role": "system",
            "content": "あなたはgithubの操作を行うAIエージェントです。"
        },
        {
            "role": "user",
            "content": "yukihito-jokyuのmcp-templateレポジトリに「mcpについて」というタイトルでissueを作成してください。"
        },
    ],
    tools=tool_list,
    tool_choice="auto"
)

In [43]:
with open("results/choice_tool.json", "w", encoding="utf-8") as f:
    json.dump(response.model_dump(), f, ensure_ascii=False, indent=2)

In [44]:
name = response.choices[0].message.tool_calls[0].function.name
args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)

In [46]:
with open("results/args.json", "w", encoding="utf-8") as f:
    json.dump(args, f, ensure_ascii=False, indent=2)

In [47]:
async with AsyncExitStack() as stack:
    server_params = StdioServerParameters(
        command='docker',
        args=[
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        env={
            "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_TOKEN", ""),
        }
    )

    stdio_transport = await stack.enter_async_context(
        stdio_client(server_params)
    )

    stdio, write = stdio_transport
    session = await stack.enter_async_context(
        ClientSession(stdio, write)
    )

    await session.initialize()

    response = await session.list_tools()

    result = await session.call_tool(name, args)


In [49]:
with open("results/tool_result.json", "w", encoding="utf-8") as f:
    json.dump(result.model_dump(), f, ensure_ascii=False, indent=2)

In [57]:
response = llm.chat.completions.create(
    model=os.getenv("DEPLOYMENT_NAME"),
    messages=[
        {
            "role": "system",
            "content": "あなたはgithubの操作を行うAIエージェントです。"
        },
        {
            "role": "user",
            "content": "yukihito-jokyuのmcp-templateレポジトリに「mcpについて」というタイトルでissueを作成してください。"
        },
        {
            "role": "system",
            "content": "```\n" + str(result.content) + "\n```",
            "metadata": {
                "parent_id": f"result_{name}",
                "id": f"raw_result_{name}",
                "title": "Raw Output"
            }
        },
        {
            "role": "user",
            "content": f"Tool result for {name}:\n{str(result.content)}"
        },
    ],
)

In [59]:
with open("results/end.json", "w", encoding="utf-8") as f:
    json.dump(response.model_dump(), f, ensure_ascii=False, indent=2)